In [1]:
from pathlib import Path
import numpy as np
import tensorflow as tf
tf.compat.v1.disable_eager_execution()
tf1 = tf.compat.v1

def load_idx_images(path):
    with open(path, 'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
    return data.reshape(-1, 28, 28)

def load_idx_labels(path):
    with open(path, 'rb') as f:
        data = np.frombuffer(f.read(), dtype=np.uint8, offset=8)
    return data

raw_dir = Path('mnist/MNIST/raw')
if raw_dir.exists():
    train_images = load_idx_images(raw_dir / 'train-images-idx3-ubyte')
    train_labels = load_idx_labels(raw_dir / 'train-labels-idx1-ubyte')
    test_images = load_idx_images(raw_dir / 't10k-images-idx3-ubyte')
    test_labels = load_idx_labels(raw_dir / 't10k-labels-idx1-ubyte')
else:
    (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape(-1, 784).astype(np.float32)
test_images = test_images.reshape(-1, 784).astype(np.float32)
train_labels = np.eye(10)[train_labels].astype(np.float32)
test_labels = np.eye(10)[test_labels].astype(np.float32)

class SimpleMNIST:
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
        self._index = 0

    def next_batch(self, batch_size):
        if self._index + batch_size > len(self.images):
            perm = np.random.permutation(len(self.images))
            self.images = self.images[perm]
            self.labels = self.labels[perm]
            self._index = 0
        start = self._index
        end = start + batch_size
        self._index = end
        return self.images[start:end], self.labels[start:end]

class MNISTWrapper:
    def __init__(self, train_images, train_labels, test_images, test_labels):
        self.train = SimpleMNIST(train_images, train_labels)
        self.test = SimpleMNIST(test_images, test_labels)
        self.test.images = test_images
        self.test.labels = test_labels

mnist = MNISTWrapper(train_images, train_labels, test_images, test_labels)

learning_rate = 1e-3
keep_prob_rate = 0.7 # 
max_epoch = 2000
def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
    correct_prediction = tf.equal(tf.argmax(y_pre,1), tf.argmax(v_ys,1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1})
    return result

def weight_variable(shape):
    initial = tf.random.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # 每一维度  滑动步长全部是 1， padding 方式 选择 same
    # 提示 使用函数  tf.nn.conv2d
    
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    # 滑动步长 是 2步; 池化窗口的尺度 高和宽度都是2; padding 方式 请选择 same
    # 提示 使用函数  tf.nn.max_pool
    
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# define placeholder for inputs to network
xs = tf1.placeholder(tf.float32, [None, 784])/255.
ys = tf1.placeholder(tf.float32, [None, 10])
keep_prob = tf1.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

#  卷积层 1
## conv1 layer ##

W_conv1 = weight_variable([7, 7, 1, 32])                      # patch 7x7, in size 1, out size 32
b_conv1 = bias_variable([32])                     
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)                      # 卷积  自己选择 选择激活函数
h_pool1 = max_pool_2x2(h_conv1)                      # 池化               

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])                       # patch 5x5, in size 32, out size 64
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)                       # 卷积  自己选择 选择激活函数
h_pool2 = max_pool_2x2(h_conv2)                       # 池化

#  全连接层 1
## fc1 layer ##
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])

h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, rate=1 - keep_prob)

# 全连接层 2
## fc2 layer ##
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
logits = tf.matmul(h_fc1_drop, W_fc2) + b_fc2
prediction = tf.nn.softmax(logits)


# 交叉熵函数
cross_entropy = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=ys, logits=logits))
train_step = tf1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf1.Session() as sess:
    init = tf1.global_variables_initializer()
    sess.run(init)
    
    for i in range(max_epoch):
        batch_xs, batch_ys = mnist.train.next_batch(100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob:keep_prob_rate})
        if i % 100 == 0:
            print(compute_accuracy(
                mnist.test.images[:1000], mnist.test.labels[:1000]))


I0000 00:00:1774120429.736758 1102011 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


0.129
0.213
0.193
0.215
0.218
0.215
0.218
0.218
0.219
0.218
0.219
0.218
0.218
0.218
0.215
0.215
0.215
0.215
0.22
0.22
